07_door_or_trap

In [ ]:
import re, warnings
warnings.filterwarnings("ignore")
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

OUT_DIR = Path("outputs")
stage2 = pd.read_pickle(OUT_DIR / "stage2_working_women.pkl").reset_index(drop=True)
VL = pd.read_pickle(OUT_DIR / "value_labels.pkl")
ORDER = ["Traditional", "Gig-proxy", "Startup-proxy"]
d = stage2.copy()

def vlab(series, var):
    """SPSS value labels for variable `var`; missing stays missing."""
    m = {}
    for k, v in VL.get(var, {}).items():
        try: m[str(int(float(k)))] = v
        except Exception: m[str(k)] = v
    s = series.astype(str).str.replace(r"\.0$", "", regex=True)
    return s.map(m).fillna(s).where(series.notna(), np.nan)
daywise = sorted(c for c in d.columns if c.startswith("b6q9_") and "3pt" in c)

print("Day-wise wage columns found:", len(daywise), "(expected 14)")
print("Day-wise wage columns found:", daywise)
display(d[daywise].apply(pd.to_numeric, errors="coerce").describe().round(1))
print("CHECK: these should look like rupee amounts per day (0 to a few thousand).")
print("CHECK: the plan mentions up to 14 columns (two activities). Only the names above exist here.")

for c in ["Total_Income", "Total_income_clean"]:
    x = pd.to_numeric(d[c], errors="coerce")
    print(f"{c}: non-missing {x.notna().mean():.3f} | >0 {(x > 0).mean():.3f} | <0 {(x < 0).mean():.3f} "
          f"| median of positives {x[x > 0].median():.0f}")
for c in ["Casual_Labour", "earning_mem"]:
    print(f"\n{c}:"); print(d[c].value_counts(dropna=False).head(8))
print("\nUsual status codes:"); print(d["usual_status"].value_counts(dropna=False))

In [ ]:
for c in ["Total_Income"] + daywise:
    d[c] = pd.to_numeric(d[c], errors="coerce")

d["weekly_casual"] = d[daywise].sum(axis=1, min_count=1)        # rupees earned in the last week
d["casual_monthly_A"] = d["weekly_casual"] * 30 / 7             # assumption A: weekly x 30/7
d["casual_monthly_B"] = d["weekly_casual"] * 4                  # assumption B: weekly x 4

for k in ["A", "B"]:
    d[f"earn_{k}"] = d["Total_Income"].fillna(0) + d[f"casual_monthly_{k}"].fillna(0)
    d.loc[d["Total_Income"].isna() & d["weekly_casual"].isna(), f"earn_{k}"] = np.nan   # nothing recorded

def earn_table(df):
    rows = {}
    for g in ORDER:
        x = df[df["own_economy_type"] == g]
        pos = x.loc[x["earn_A"] > 0, "earn_A"]
        rows[g] = {
            "n": len(x),
            "Total_Income > 0": (x["Total_Income"] > 0).mean(),
            "Total_Income < 0 (losses)": (x["Total_Income"] < 0).mean(),
            "any earnings incl. casual wages (A)": (x["earn_A"] > 0).mean(),
            "any earnings incl. casual wages (B)": (x["earn_B"] > 0).mean(),
            "median earnings of earners, A (Rs/month)": pos.median(),
            "25th percentile": pos.quantile(0.25), "75th percentile": pos.quantile(0.75),
            "median of earners, B (Rs/month)": x.loc[x["earn_B"] > 0, "earn_B"].median(),
        }
    return pd.DataFrame(rows)

earn = earn_table(d)
display(earn.round(3)); earn.round(3).to_csv(OUT_DIR / "dot_earnings.csv")

In [ ]:
all_w = pd.read_pickle(OUT_DIR / "stage1_all_women.pkl")

def pop_weight(df_):
    need = ["NSS_perv1", "NSC_perv1", "mult_perv1"]
    if not set(need) <= set(df_.columns):
        return pd.Series(1.0, index=df_.index)
    nss, nsc, mult = [pd.to_numeric(df_[c], errors="coerce") for c in need]
    return pd.Series(np.where(nss == nsc, mult / 100, mult / 200), index=df_.index).fillna(0)

def wquant(x, w, q):
    x, w = np.asarray(x, float), np.asarray(w, float)
    m = ~np.isnan(x) & ~np.isnan(w)
    x, w = x[m], w[m]
    o = np.argsort(x); x, w = x[o], w[o]
    cw = np.cumsum(w) - 0.5 * w
    return float(np.interp(q * w.sum(), cw, x))

pc_all = pd.to_numeric(all_w["b3q5pt6_hhv1"], errors="coerce") / pd.to_numeric(all_w["b3q1_hhv1"], errors="coerce")
cuts = [wquant(pc_all, pop_weight(all_w), q) for q in (0.2, 0.4, 0.6, 0.8)]
print("Quintile cut points (rupees per person per month, all women 15-59):", [round(c) for c in cuts])

d["pc_exp"] = pd.to_numeric(d["b3q5pt6_hhv1"], errors="coerce") / pd.to_numeric(d["b3q1_hhv1"], errors="coerce")
d["pc_exp_quintile"] = pd.cut(d["pc_exp"], [-np.inf] + cuts + [np.inf],
                              labels=["Q1 (poorest)", "Q2", "Q3", "Q4", "Q5 (richest)"])
d["w_pop"] = pop_weight(d)

q = pd.crosstab(d["pc_exp_quintile"], d["own_economy_type"], values=d["w_pop"], aggfunc="sum",
                normalize="columns")[ORDER].round(3)
print("Weighted share of each economy type's women in each household expenditure quintile (quintiles from ALL women):")
display(q); q.to_csv(OUT_DIR / "dot_expenditure_quintiles.csv")

q.plot.bar(figsize=(8, 4), color=["#1F3A5F", "#B5483A", "#7F9BB8"])
plt.ylabel("Share within economy type"); plt.title("Household welfare by economy type")
plt.tight_layout(); plt.savefig(OUT_DIR / "fig_dot_expenditure.png", dpi=150); plt.show()

In [ ]:
d["ss_label"] = vlab(d["Social_Security"], "Social_Security")
d["contract_label"] = vlab(d["b5pt1q11_perv1"], "b5pt1q11_perv1")
print(d["ss_label"].value_counts(dropna=False)); print(d["contract_label"].value_counts(dropna=False))

lab_ss = d["ss_label"].astype(str).str.lower()
d["ss_recorded"] = d["ss_label"].notna() & ~lab_ss.str.contains("not known")
d["ss_none"] = d["ss_recorded"] & lab_ss.str.contains("none")
lab_c = d["contract_label"].astype(str).str.lower()
d["contract_recorded"] = d["contract_label"].notna()
d["has_written"] = d["contract_recorded"] & ~lab_c.str.contains("no written")
if d["ss_none"].sum() == 0:
    print("WARNING: no 'None' label found for Social_Security. Check the printed labels above and set ss_none by hand.")

In [ ]:
def scorecard(df):
    rows = {}
    for g in ORDER:
        x = df[df["own_economy_type"] == g]
        rows[g] = {
            "n": len(x),
            "rural share": (x["sector"] == "1").mean(),
            "any earnings incl. casual (A)": (x["earn_A"] > 0).mean(),
            "median monthly earnings, earners (A, Rs)": x.loc[x["earn_A"] > 0, "earn_A"].median(),
            "poorest 40% of households": x["pc_exp_quintile"].isin(["Q1 (poorest)", "Q2"]).mean(),
            "social security recorded": x["ss_recorded"].mean(),
            "no social security (of recorded)": x.loc[x["ss_recorded"], "ss_none"].mean() if x["ss_recorded"].any() else np.nan,
            "contract recorded": x["contract_recorded"].mean(),
            "written contract (of recorded)": x.loc[x["contract_recorded"], "has_written"].mean() if x["contract_recorded"].any() else np.nan,
        }
    return pd.DataFrame(rows)

score = scorecard(d)
display(score.round(3)); score.round(4).to_csv(OUT_DIR / "dot_scorecard.csv")

fig, ax = plt.subplots(figsize=(8, 4))
data = [d.loc[(d["own_economy_type"] == g) & (d["earn_A"] > 0), "earn_A"].clip(upper=d["earn_A"].quantile(0.99)) for g in ORDER]
ax.boxplot(data, labels=ORDER, showfliers=False)
ax.set_ylabel("Monthly earnings, earners only (Rs, assumption A)"); ax.set_title("Earnings by economy type")
plt.tight_layout(); plt.savefig(OUT_DIR / "fig_dot_earnings.png", dpi=150); plt.show()

In [ ]:
STATUS = {"11": "Own-account worker", "41": "Casual labour, public works", "51": "Casual labour, other"}
d["status_code"] = d["usual_status"].astype(str).str.replace(r"\.0$", "", regex=True)

def diff_ci(k1, n1, k0, n0):
    if n1 == 0 or n0 == 0: return (np.nan,) * 5
    p1, p0 = k1 / n1, k0 / n0
    se = np.sqrt(p1 * (1 - p1) / n1 + p0 * (1 - p0) / n0)
    return p1, p0, p1 - p0, p1 - p0 - 1.96 * se, p1 - p0 + 1.96 * se

always = lambda x: pd.Series(True, index=x.index)
metrics = {
    "any earnings incl. casual (A)": (lambda x: x["earn_A"] > 0, always),
    "poorest 40% of households": (lambda x: x["pc_exp_quintile"].isin(["Q1 (poorest)", "Q2"]), lambda x: x["pc_exp_quintile"].notna()),
    "no social security | recorded": (lambda x: x["ss_none"], lambda x: x["ss_recorded"]),
    "written contract | recorded": (lambda x: x["has_written"], lambda x: x["contract_recorded"]),
}
rows = []
for code, name in STATUS.items():
    sub = d[d["status_code"] == code]
    gig, trad = sub[sub["own_economy_type"] == "Gig-proxy"], sub[sub["own_economy_type"] == "Traditional"]
    for mname, (num_f, den_f) in metrics.items():
        k1, n1 = int((num_f(gig) & den_f(gig)).sum()), int(den_f(gig).sum())
        k0, n0 = int((num_f(trad) & den_f(trad)).sum()), int(den_f(trad).sum())
        p1, p0, diff, lo, hi = diff_ci(k1, n1, k0, n0)
        rows.append({"status group": name, "measure": mname, "n gig": n1, "n traditional": n0,
                     "gig share": p1, "traditional share": p0, "difference": diff, "95% CI low": lo, "95% CI high": hi})
lfl = pd.DataFrame(rows).round(3)
display(lfl); lfl.to_csv(OUT_DIR / "dot_like_for_like.csv", index=False)

med = (d[d["earn_A"] > 0].loc[d["status_code"].isin(STATUS)]
       .groupby(["status_code", "own_economy_type"])["earn_A"].agg(["size", "median"]).round(0))
print("Median monthly earnings of earners within status groups (assumption A):"); display(med)
print("Rows with CI containing 0 show no clear difference. CIs ignore household clustering, so treat them as rough.")

In [ ]:
def boot_prop_diff(sub, num, den, reps=2000, seed=42):
    """Gig share minus traditional share, with a 95% interval from resampling households."""
    g = (sub["own_economy_type"] == "Gig-proxy").to_numpy()
    num = np.asarray(num, float); den = np.asarray(den, float)
    codes, _ = pd.factorize(sub["HHID"])
    m_ = codes.max() + 1
    H = np.zeros((m_, 4))
    np.add.at(H[:, 0], codes, num * den * g)
    np.add.at(H[:, 1], codes, den * g)
    np.add.at(H[:, 2], codes, num * den * (~g))
    np.add.at(H[:, 3], codes, den * (~g))
    tot = H.sum(axis=0)
    if tot[1] == 0 or tot[3] == 0:
        return None
    point = tot[0] / tot[1] - tot[2] / tot[3]
    rng = np.random.default_rng(seed)
    draws = []
    for _ in range(reps):
        s = H[rng.integers(0, m_, m_)].sum(axis=0)
        if s[1] > 0 and s[3] > 0:
            draws.append(s[0] / s[1] - s[2] / s[3])
    lo, hi = np.percentile(draws, [2.5, 97.5])
    return tot[0] / tot[1], tot[2] / tot[3], point, lo, hi, int(tot[1]), int(tot[3])

rows = []
for code_, name in STATUS.items():
    sub = d[(d["status_code"] == code_) & d["own_economy_type"].isin(["Gig-proxy", "Traditional"])]
    for mname, (num_f, den_f) in metrics.items():
        r_ = boot_prop_diff(sub, num_f(sub).astype(float).fillna(0).to_numpy(), den_f(sub).astype(float).fillna(0).to_numpy())
        if r_ is None:
            continue
        rows.append({"status group": name, "measure": mname, "n gig": r_[5], "n traditional": r_[6],
                     "gig share": r_[0], "traditional share": r_[1], "difference": r_[2],
                     "95% CI low": r_[3], "95% CI high": r_[4],
                     "clearly different": (r_[3] > 0) or (r_[4] < 0)})
lfl_c = pd.DataFrame(rows).round(3)
display(lfl_c); lfl_c.to_csv(OUT_DIR / "dot_like_for_like_clustered.csv", index=False)

def boot_median_diff(sub, col, reps=500, seed=42):
    s = sub[sub[col] > 0]
    x = s[col].to_numpy(float); g = (s["own_economy_type"] == "Gig-proxy").to_numpy()
    codes, _ = pd.factorize(s["HHID"])
    order = np.argsort(codes, kind="stable"); counts = np.bincount(codes)
    starts = np.r_[0, np.cumsum(counts)[:-1]]
    idx = [order[a:a + c] for a, c in zip(starts, counts)]
    def diff(rows_):
        xr, gr = x[rows_], g[rows_]
        if gr.sum() == 0 or (~gr).sum() == 0:
            return np.nan
        return np.median(xr[gr]) - np.median(xr[~gr])
    point = diff(np.arange(len(x)))
    rng = np.random.default_rng(seed)
    draws = [diff(np.concatenate([idx[k] for k in rng.integers(0, len(idx), len(idx))])) for _ in range(reps)]
    lo, hi = np.nanpercentile(draws, [2.5, 97.5])
    return point, lo, hi, int(g.sum()), int((~g).sum())

mrows = []
for code_, name in STATUS.items():
    sub = d[(d["status_code"] == code_) & d["own_economy_type"].isin(["Gig-proxy", "Traditional"])]
    p_, lo_, hi_, n1, n0 = boot_median_diff(sub, "earn_A")
    mrows.append({"status group": name, "n gig earners": n1, "n traditional earners": n0,
                  "median difference (Rs/month)": p_, "95% CI low": lo_, "95% CI high": hi_,
                  "clearly different": (lo_ > 0) or (hi_ < 0)})
med_c = pd.DataFrame(mrows).round(0)
print("Gig-proxy minus traditional, median monthly earnings of earners (assumption A):")
display(med_c); med_c.to_csv(OUT_DIR / "dot_earnings_median_diff_clustered.csv", index=False)